## Clase 7 - Ejercicio: algoritmo de búsqueda para la Torre de Hanoi

**Rodeyro, Agustín**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

---

Elegí **A\* con una heurística propia**, que es la opción de nota 10.

La idea de la heurística es calcular el mínimo exacto de movimientos que faltan hasta el
objetivo, en lugar de una estimación gruesa. Abajo explico de dónde sale, verifico sobre
los 243 estados del problema que es válida, y comparo cuántos nodos hay que explorar con
ella y sin ella.

Encuentra la solución óptima de 31 movimientos explorando 32 nodos. Con la heurística del
aula son 178, y con costo uniforme 233.

In [1]:
import heapq
import itertools

from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

### Las dos heurísticas

`h(n)` estima cuántos movimientos faltan desde el nodo `n` hasta el objetivo. Para que A\*
devuelva el camino óptimo tiene que ser admisible, o sea que nunca se pase del costo real.

Implementé dos, para poder comparar.

La **del aula virtual** suma un punto por cada disco mal ubicado, contando los que todavía
no están en la tercera varilla. Es admisible, porque cada uno de esos discos va a necesitar
por lo menos un movimiento, pero informa poco: como mucho vale 5 cuando faltan 31
movimientos.

#### De dónde sale la heurística propia

Me puse a pensar qué pasa con el disco más grande, el `k`. Llamo `d(k, t)` al mínimo de
movimientos para llevar los discos `1..k` a la varilla `t`. Se dan dos situaciones.

Si el disco `k` **ya está en `t`**, no tiene sentido moverlo: sacarlo y volverlo a poner
solo suma movimientos. El problema queda reducido a los discos más chicos sobre esa misma
varilla.

`d(k, t) = d(k-1, t)`

Si **está en otra varilla `p`**, para moverlo a `t` los `k-1` discos de arriba tienen que
estar todos apilados en la tercera varilla, `o = 3 - p - t`, que es la única donde no
molestan. Entonces hay que hacer tres cosas sí o sí: llevar los chicos a `o`, que cuesta
`d(k-1, o)`; mover el disco `k`, que cuesta 1; y después pasar la torre de `k-1` discos de
`o` a `t`, que es una Hanoi completa y cuesta `2^(k-1) - 1`.

`d(k, t) = d(k-1, o) + 1 + (2^(k-1) - 1) = d(k-1, o) + 2^(k-1)`

Lo importante es que esas tres cosas son obligatorias y no se pisan entre sí, así que la
suma da el mínimo y no una cota floja.

Desplegando la recursión termina no haciendo falta recursión: alcanza con recorrer los
discos de mayor a menor sumando `2^(k-1)` cada vez que el disco `k` no está donde va, y en
ese caso pasar a mirar la tercera varilla.

In [2]:
def h_aula(state: StatesHanoi, number_disks: int) -> int:
    """Heuristica del aula virtual: un punto por cada disco mal ubicado.

    Equivale a la version no negativa h*(n) = h(n) + number_disks del apunte.
    """
    return number_disks - len(state.rods[2])


def h_propia(state: StatesHanoi, number_disks: int) -> int:
    """Heuristica propia: minimo exacto de movimientos que faltan hasta el objetivo.

    Recorre los discos del mas grande al mas chico. Cada vez que un disco no esta en la
    varilla donde deberia, ese disco tendra que moverse y ademas obliga a todos los mas
    chicos a pasar por la tercera varilla, lo que cuesta 2**(disco-1) movimientos.
    """
    donde = {}
    for indice, varilla in enumerate(state.rods):
        for disco in varilla:
            donde[disco] = indice

    total = 0
    objetivo = 2                                      # la tercera varilla
    for disco in range(number_disks, 0, -1):          # del mas grande al mas chico
        if donde[disco] != objetivo:
            total += 2 ** (disco - 1)                 # 1 + (2**(disco-1) - 1)
            objetivo = 3 - donde[disco] - objetivo    # la tercera varilla
    return total


# Control rapido sobre el estado inicial y el objetivo
_discos = [i for i in range(5, 0, -1)]
_inicial = StatesHanoi(_discos, [], [], max_disks=5)
_objetivo = StatesHanoi([], [], _discos, max_disks=5)

print(f"{'estado':<32}{'h_aula':>8}{'h_propia':>10}")
print(f"{str(_inicial):<32}{h_aula(_inicial, 5):>8}{h_propia(_inicial, 5):>10}")
print(f"{str(_objetivo):<32}{h_aula(_objetivo, 5):>8}{h_propia(_objetivo, 5):>10}")
print(f"\nCosto real de la solucion optima: {2 ** 5 - 1}")
print("La heuristica propia lo acierta exactamente desde el estado inicial.")

estado                            h_aula  h_propia
HanoiState: 5 4 3 2 1 |  |             5        31
HanoiState:  |  | 5 4 3 2 1            0         0

Costo real de la solucion optima: 31
La heuristica propia lo acierta exactamente desde el estado inicial.


### El algoritmo

A\* ordena la frontera por `f(n) = g(n) + h(n)`, con `g(n)` el costo real acumulado hasta
el nodo y `h(n)` lo que estimo que falta.

Dos detalles de la biblioteca del aula. El costo ya viene acumulado en el estado, así
que `nodo.path_cost` es directamente `g(n)`. Y le agregué un `itertools.count()` como
segundo criterio de orden, para que cuando dos nodos empatan en `f` el desempate sea
siempre el mismo y el resultado no cambie entre corridas.

In [3]:
def search_algorithm(number_disks=5, heuristic=h_propia) -> tuple[NodeHanoi | None, dict]:

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    ##### EDITAR ESTA ZONA

    # Busqueda A*: f(n) = g(n) + h(n)
    desempate = itertools.count()
    raiz = NodeHanoi(problem.initial)

    # La frontera es una cola de prioridad ordenada por f(n)
    frontera = [(raiz.path_cost + heuristic(raiz.state, number_disks),
                 next(desempate), raiz)]

    # Mejor costo conocido para llegar a cada estado, para no reabrir caminos peores
    mejor_costo = {raiz.state: raiz.path_cost}
    explorados = set()

    nodes_explored = 0
    max_depth = 0

    while frontera:
        _, _, nodo = heapq.heappop(frontera)

        # Puede haber quedado una entrada vieja del mismo estado en la cola
        if nodo.state in explorados:
            continue

        explorados.add(nodo.state)
        nodes_explored += 1
        max_depth = max(max_depth, nodo.depth)

        if problem.goal_test(nodo.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": nodes_explored,
                "states_visited": len(explorados),
                "nodes_in_frontier": len(frontera),
                "max_depth": max_depth,
                "cost_total": float(nodo.state.accumulated_cost),
            }
            return nodo, metrics

        for hijo in nodo.expand(problem):
            # Solo vale la pena encolarlo si llegamos al estado por un camino mas barato
            if hijo.path_cost < mejor_costo.get(hijo.state, float("inf")):
                mejor_costo[hijo.state] = hijo.path_cost
                heapq.heappush(
                    frontera,
                    (hijo.path_cost + heuristic(hijo.state, number_disks),
                     next(desempate), hijo),
                )

    # Se agoto la frontera sin llegar al objetivo
    metrics = {
        "solution_found": False,
        "nodes_explored": nodes_explored,
        "states_visited": len(explorados),
        "nodes_in_frontier": len(frontera),
        "max_depth": max_depth,
        "cost_total": 0.0,
    }
    return None, metrics

    #####

Se prueba la función:

In [4]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [5]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 32
states_visited: 32
nodes_in_frontier: 16
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [6]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [7]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3


---

## Verificación

El algoritmo devuelve 31 movimientos, pero eso solo no prueba nada. Me faltaba chequear
tres cosas: que la secuencia se pueda jugar de verdad, que la heurística sea válida, y si
realmente sirvió de algo.

### Que la secuencia sea jugable

Recorro el camino paso por paso y me fijo que cada movimiento esté entre las acciones
legales del problema, que no se mueva más de un disco por vez, y que arranque y termine
donde tiene que ser.

In [8]:
def validar(solution, number_disks=5):
    camino = solution.path()
    list_disks = [i for i in range(number_disks, 0, -1)]
    inicial = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    objetivo = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=inicial, goal=objetivo)

    assert camino[0].state == inicial, "no arranca en el estado inicial"
    assert camino[-1].state == objetivo, "no termina en el estado objetivo"

    for anterior, siguiente in zip(camino, camino[1:]):
        legales = [problem.result(anterior.state, acc)
                   for acc in problem.actions(anterior.state)]
        assert siguiente.state in legales, f"movimiento ilegal: {anterior.state}"

    for accion in solution.solution():
        assert accion.cost == 1.0, "hay una accion que no es un movimiento simple"

    return True


print("Secuencia valida:", validar(solution))
print(f"Movimientos: {len(solution.solution())}")
print(f"Costo total: {solution.state.accumulated_cost}")
print(f"Optimo teorico 2^5 - 1: {2 ** 5 - 1}")

Secuencia valida: True
Movimientos: 31
Costo total: 31.0
Optimo teorico 2^5 - 1: 31


### Que la heurística sea válida

Genero los 243 estados del problema, que son las `3^5` maneras de repartir 5 discos en 3
varillas, y chequeo las propiedades sobre todos en vez de sobre un par de ejemplos.

Alcanza con verificar la consistencia, `h(n) <= costo(n, a, n') + h(n')`, más que
`h(objetivo)` valga 0. Por el teorema que vimos en clase, con eso ya queda admisible.

In [9]:
def todos_los_estados(number_disks=5):
    """Genera los 3**number_disks estados posibles como objetos StatesHanoi."""
    estados = []
    for asignacion in itertools.product(range(3), repeat=number_disks):
        varillas = ([], [], [])
        for disco in range(number_disks, 0, -1):      # de grande a chico: la base primero
            varillas[asignacion[disco - 1]].append(disco)
        estados.append(StatesHanoi(*varillas, max_disks=number_disks))
    return estados


estados = todos_los_estados(5)
print(f"Estados generados: {len(estados)}   (3^5 = {3 ** 5})")

Estados generados: 243   (3^5 = 243)


In [10]:
def revisar(h, nombre, estados, number_disks=5):
    list_disks = [i for i in range(number_disks, 0, -1)]
    problem = ProblemHanoi(
        initial=StatesHanoi(list_disks, [], [], max_disks=number_disks),
        goal=StatesHanoi([], [], list_disks, max_disks=number_disks),
    )
    objetivo = problem.goal

    inconsistentes = 0
    for estado in estados:
        for accion in problem.actions(estado):
            siguiente = problem.result(estado, accion)
            if h(estado, number_disks) > accion.cost + h(siguiente, number_disks):
                inconsistentes += 1

    print(nombre)
    print(f"   h(objetivo) = 0                  : {h(objetivo, number_disks) == 0}")
    print(f"   consistente en toda transicion   : {inconsistentes == 0}"
          f"   [{inconsistentes} violaciones]")
    print(f"   admisible (por el teorema)       : "
          f"{inconsistentes == 0 and h(objetivo, number_disks) == 0}")


revisar(h_aula, "h_aula (heuristica del aula virtual)", estados)
print()
revisar(h_propia, "h_propia (heuristica propia)", estados)

h_aula (heuristica del aula virtual)
   h(objetivo) = 0                  : True
   consistente en toda transicion   : True   [0 violaciones]
   admisible (por el teorema)       : True

h_propia (heuristica propia)
   h(objetivo) = 0                  : True
   consistente en toda transicion   : True   [0 violaciones]
   admisible (por el teorema)       : True


Las dos pasan, así que A\* devuelve el óptimo con cualquiera de las dos. Lo que cambia es
cuánto informan.

Para medirlo necesito la distancia real de cada estado al objetivo. La saco con un costo
uniforme hacia atrás arrancando desde el objetivo, que se puede hacer porque todos los
movimientos son reversibles y cuestan lo mismo. Esto es solo para verificar, no es el
algoritmo que entrego.

In [11]:
def distancias_reales(number_disks=5):
    """Costo uniforme hacia atras desde el objetivo. Solo para verificar la heuristica."""
    list_disks = [i for i in range(number_disks, 0, -1)]
    objetivo = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=objetivo, goal=objetivo)

    desempate = itertools.count()
    frontera = [(0.0, next(desempate), objetivo)]
    distancia = {objetivo: 0.0}
    while frontera:
        costo, _, estado = heapq.heappop(frontera)
        if costo > distancia[estado]:
            continue
        for accion in problem.actions(estado):
            vecino = problem.result(estado, accion)
            if costo + accion.cost < distancia.get(vecino, float("inf")):
                distancia[vecino] = costo + accion.cost
                heapq.heappush(frontera, (distancia[vecino], next(desempate), vecino))
    return distancia


real = distancias_reales(5)
print(f"Estados etiquetados: {len(real)}")

for h, nombre in [(h_aula, "h_aula  "), (h_propia, "h_propia")]:
    exactos = sum(1 for e, d in real.items() if h(e, 5) == d)
    sobre = sum(1 for e, d in real.items() if h(e, 5) > d)
    print(f"{nombre}: coincide con la distancia real en {exactos:>3} de {len(real)} estados"
          f"  |  la sobreestima en {sobre}")

Estados etiquetados: 243
h_aula  : coincide con la distancia real en   5 de 243 estados  |  la sobreestima en 0
h_propia: coincide con la distancia real en 243 de 243 estados  |  la sobreestima en 0


Ahí se ve la diferencia. La del aula acierta en 5 estados y en el resto se queda corta, por
eso orienta tan poco. La mía coincide con la distancia real en los 243: no es una
estimación, es la distancia exacta.

Eso explica lo que pasa después. Si `h` es la distancia real, `f(n) = g(n) + h(n)` termina
siendo el costo del mejor camino que pasa por `n`. Todos los nodos del camino óptimo tienen
el mismo `f`, y cualquier nodo que se salga de ese camino tiene un `f` más grande. Por eso
A\* no se desvía y explora solo el camino, más el objetivo.

### Si valió la pena

In [12]:
comparacion = [
    ("Costo uniforme (h = 0)", lambda s, n: 0),
    ("A* + heuristica del aula", h_aula),
    ("A* + heuristica propia", h_propia),
]

print(f"{'algoritmo':<26}{'movs':>6}{'optima':>9}{'explorados':>12}"
      f"{'frontera':>10}{'costo':>8}")
print("-" * 71)
for nombre, h in comparacion:
    sol, met = search_algorithm(number_disks=5, heuristic=h)
    print(f"{nombre:<26}{len(sol.solution()):>6}"
          f"{('si' if met['cost_total'] == 31.0 else 'NO'):>9}"
          f"{met['nodes_explored']:>12}{met['nodes_in_frontier']:>10}"
          f"{met['cost_total']:>8.1f}")

algoritmo                   movs   optima  explorados  frontera   costo
-----------------------------------------------------------------------
Costo uniforme (h = 0)        31       si         233        10    31.0
A* + heuristica del aula      31       si         178        15    31.0
A* + heuristica propia        31       si          32        16    31.0


Los tres llegan a los 31 movimientos, porque las tres heurísticas son admisibles. La
diferencia está en el trabajo que hacen.

Costo uniforme explora 233 de los 243 estados, o sea que sin heurística A\* termina
revisando casi todo el espacio. La del aula baja a 178: ayuda, pero poco, porque como mucho
vale 5 cuando faltan 31 movimientos y casi no distingue un estado de otro. La mía explora
32, que son los 31 del camino más el objetivo. Ni un nodo de más.

### Cómo escala

Probé con más discos para ver si se mantiene.

In [13]:
print(f"{'discos':>7}{'optimo':>8}{'explorados':>12}{'2^n':>7}"
      f"{'aula':>8}{'costo unif.':>13}")
print("-" * 55)
for n in range(3, 7):
    _, m_propia = search_algorithm(number_disks=n, heuristic=h_propia)
    _, m_aula = search_algorithm(number_disks=n, heuristic=h_aula)
    _, m_ucs = search_algorithm(number_disks=n, heuristic=lambda s, k: 0)
    print(f"{n:>7}{int(m_propia['cost_total']):>8}{m_propia['nodes_explored']:>12}"
          f"{2 ** n:>7}{m_aula['nodes_explored']:>8}{m_ucs['nodes_explored']:>13}")

 discos  optimo  explorados    2^n    aula  costo unif.
-------------------------------------------------------
      3       7           8      8      18           25
      4      15          16     16      54           71
      5      31          32     32     178          233
      6      63          64     64     586          687


Mientras el espacio de estados crece como `3^n` y los otros dos métodos crecen con él, la
heurística propia se queda en `2^n` exacto, que es la cantidad de nodos del camino que hay
que devolver. Menos que eso no se puede.

Para cerrar: entrego A\* con heurística propia, encuentra el óptimo de 31 movimientos, la
heurística quedó verificada sobre los 243 estados y el efecto se mide en 32 nodos contra
178 y 233. La heurística no cambia el resultado, cambia lo que cuesta llegar.